# Birddog quickstart

Notebooks here connect to the Birddog Postgres database as **`birddog_reader`**: read-only, schemas `core` and `raw` only (no `app`). The connection URL is in the environment as `BIRDDOG_DATABASE_URL`.

This notebook is a copy in your own work folder; edit freely. Delete it to get a fresh copy on your next server start.

In [ ]:
import os
DB_URL = os.environ["BIRDDOG_DATABASE_URL"]

## 1. psycopg

Use `autocommit=True` (or close your connections): an open transaction holds locks that can block imports, and idle transactions are killed after 10 minutes.

In [ ]:
import psycopg

with psycopg.connect(DB_URL, autocommit=True) as conn:
    n_pages = conn.execute("SELECT count(*) FROM core.page").fetchone()[0]
    tables = conn.execute(
        "SELECT table_schema, table_name, table_type FROM information_schema.tables"
        " WHERE table_schema IN ('core', 'raw') ORDER BY 1, 2"
    ).fetchall()

print(f"core.page rows: {n_pages:,}")
for t in tables:
    print(*t)

## 2. DuckDB over Postgres

DuckDB's `postgres` extension attaches the database read-only; queries are pushed down to Postgres and results come back as DuckDB relations (to pandas / Polars / Arrow).

In [ ]:
import duckdb

con = duckdb.connect()
con.sql("LOAD postgres")
con.sql(f"ATTACH '{DB_URL}' AS bd (TYPE postgres, READ_ONLY)")
con.sql("SELECT count(*) AS pages FROM bd.core.page").show()

In [ ]:
# a sample of pages, as a Polars DataFrame
df = con.sql("SELECT * FROM bd.core.page LIMIT 1000").pl()
df.head()

Tip: for repeated analysis, copy what you need into a local DuckDB file in your work folder, then query it without touching Postgres:

```python
local = duckdb.connect("/home/jovyan/work/birddog.duckdb")
local.sql(f"ATTACH '{DB_URL}' AS bd (TYPE postgres, READ_ONLY)")
local.sql("CREATE OR REPLACE TABLE page AS SELECT * FROM bd.core.page")
```

## 3. SQL magic (jupysql)

`DATABASE_URL` is the same read-only role in SQLAlchemy form.

In [ ]:
from sqlalchemy import create_engine

engine = create_engine(os.environ["DATABASE_URL"])
%load_ext sql
%sql engine
%sql SELECT count(*) FROM core.page

## 4. The birddog library

The repo's `birddog` package is mounted read-only at `~/birddog-lib` and on `PYTHONPATH`.

In [ ]:
import birddog, os
print(birddog.__file__)
print(sorted(m for m in os.listdir(os.path.dirname(birddog.__file__)) if m.endswith('.py'))[:10])